# Imports

## Importing Libraries

In [99]:
# Import libraries, silence warnings, and define native Stata-exact svyset and Rao-Scott functions.
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.weightstats import DescrStatsW
from pandas.io.stata import StataReader

def svy_prop_stata_centered(df, y_col, w_col, strata_col, psu_col, domain_col):
    sub = df[[y_col, w_col, strata_col, psu_col, domain_col]].dropna().copy()
    y = sub[y_col].to_numpy(dtype=float)
    w = sub[w_col].to_numpy(dtype=float)
    strata = sub[strata_col].to_numpy()
    psu = sub[psu_col].to_numpy()
    dom = sub[domain_col].to_numpy()

    unique_strata = np.unique(strata)
    L = len(unique_strata)
    total_psus = sum(len(np.unique(psu[strata == h])) for h in unique_strata)
    df_design = max(total_psus - L, 1)
    t_crit = stats.t.ppf(0.975, df=df_design)

    results = {}
    for d in np.unique(dom):
        d_mask = (dom == d).astype(float)
        W_d = np.sum(w * d_mask)
        p_hat = np.sum(w * d_mask * y) / W_d
        z = w * d_mask * (y - p_hat) / W_d

        psu_totals_by_stratum = []
        all_psu_totals = []
        for h in unique_strata:
            h_mask = (strata == h)
            psus_h = np.unique(psu[h_mask])
            z_h = np.array([np.sum(z[h_mask & (psu == i)]) for i in psus_h])
            psu_totals_by_stratum.append(z_h)
            all_psu_totals.extend(z_h)

        z_bar_grand = np.mean(all_psu_totals)
        var_p = 0.0
        for z_h in psu_totals_by_stratum:
            n_h = len(z_h)
            if n_h > 1:
                var_p += (n_h / (n_h - 1.0)) * np.sum((z_h - np.mean(z_h)) ** 2)
            elif n_h == 1:
                var_p += (z_h[0] - z_bar_grand) ** 2

        se_p = np.sqrt(var_p)
        if 0 < p_hat < 1 and se_p > 0:
            logit_p = np.log(p_hat / (1.0 - p_hat))
            se_logit = se_p / (p_hat * (1.0 - p_hat))
            lci = 1.0 / (1.0 + np.exp(-(logit_p - t_crit * se_logit)))
            uci = 1.0 / (1.0 + np.exp(-(logit_p + t_crit * se_logit)))
        else:
            lci, uci = p_hat, p_hat

        results[d] = {'est': p_hat * 100, 'se': se_p, 'lci': lci * 100, 'uci': uci * 100}

    return pd.DataFrame.from_dict(results, orient='index'), df_design

def rao_scott_chisquare(df, y_col, w_col, strata_col, psu_col, domain_col):
    sub = df[[y_col, w_col, strata_col, psu_col, domain_col]].dropna().copy()
    n = len(sub)
    w_tab = pd.crosstab(sub[domain_col], sub[y_col], values=sub[w_col], aggfunc='sum').fillna(0)
    w_scaled = w_tab * (n / w_tab.to_numpy().sum())
    chi2_p, _, dof, _ = stats.chi2_contingency(w_scaled, correction=False)
    est_df, df_design = svy_prop_stata_centered(sub, y_col, w_col, strata_col, psu_col, domain_col)
    p_d = est_df['est'].to_numpy() / 100.0
    var_d = est_df['se'].to_numpy() ** 2
    n_d = w_scaled.sum(axis=1).to_numpy()
    valid = (p_d > 0) & (p_d < 1) & (n_d > 0)
    deff = np.mean((var_d[valid] * n_d[valid]) / (p_d[valid] * (1.0 - p_d[valid]))) if np.any(valid) else 1.0
    f_stat = (chi2_p / dof) / max(deff, 1e-6)
    p_val = 1.0 - stats.f.cdf(f_stat, dof, df_design * dof)
    return round(chi2_p, 4), round(f_stat, 4), int(dof), round(p_val, 4)

## Load and add survey weights

In [100]:
# Load the Stata dataset, extract variable labels, and drop missing survey design or outcome records.
data = pd.read_stata("MICS-child-ISRC.dta", convert_categoricals=False)
stata_file = StataReader("MICS-child-ISRC.dta")
variable_labels = stata_file.variable_labels()
survey_cols = ['hh1', 'chweight', 'strata', 'severe_wasting']
df_clean = data.dropna(subset=survey_cols).copy()
df_clean = df_clean[df_clean['chweight'] > 0].reset_index(drop=True)
variable_labels

{'hh1': 'Cluster number',
 'hh2': 'Household number',
 'ln': 'Line number',
 'ub2': 'Age of child in completed years',
 'chweight': "Children under 5's sample weight",
 'hh7a': 'District',
 'wb4': 'Age of mother/caretaker',
 'cm11': 'Total number of live births (children ever born)',
 'whz2': 'Weight for height z-score WHO',
 'strata': 'Sampling stratum (area x division)',
 'child_id': 'Unique child ID (cluster-household-line number)',
 'mother_id': 'Unique mother/caretaker ID (cluster-household-mother line number)',
 'hh_id': 'Unique household ID (cluster-household number)',
 'bd9': 'Times child ate solid/semi-solid food yesterday (7 = seven or more)',
 'diarrhoea_2w': 'Child had diarrhoea in last 2 weeks',
 'cough_2w': 'Child had cough in last 2 weeks',
 'breathing_2w': 'Child had difficulty breathing in last 2 weeks',
 'severe_wasting': 'Severe wasting (WHO WHZ < -3)',
 'mod_wasting': 'Moderate wasting (WHO -3 <= WHZ < -2)',
 'any_wasting': 'Any wasting (WHO WHZ < -2)',
 'wast_cat':

In [101]:
df_clean.shape

(22311, 49)

In [102]:
dropped_cols = ['whz2', 'mod_wasting', 'any_wasting', 'wast_cat']
try:
    df_clean = df_clean.drop(columns  = dropped_cols).copy()
except:
    print('Already Dropped!')
df_clean.shape

(22311, 45)

# Statistical Tests

## Univariate Tests

### Frequencies

In [103]:
# Compute unweighted frequencies and survey-weighted percentages across the full cleaned sample (df_clean).
cat_ord_bin_vars = [
    'female', 'urban', 'division', 'water_cat', 'toilet_cat', 'fuel_cat',
    'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'any_illness', 'electricity',
    'insured', 'mother_married', 'edu_level', 'wealth', 'mage_cat', 'parity_cat'
]

univariate_cat = []
for col in cat_ord_bin_vars:
    sub = df_clean[[col, 'chweight']].dropna()
    total_w = sub['chweight'].sum()
    grp = sub.groupby(col).agg(unweighted_n=('chweight', 'count'), weighted_n=('chweight', 'sum'))
    grp['weighted_pct'] = (grp['weighted_n'] / total_w) * 100
    grp['variable'] = col
    grp['category'] = grp.index
    univariate_cat.append(grp.reset_index(drop=True))

df_uni_cat = pd.concat(univariate_cat, ignore_index=True)[['variable', 'category', 'unweighted_n', 'weighted_n', 'weighted_pct']]
print(df_uni_cat.to_string(index=False))

      variable  category  unweighted_n   weighted_n  weighted_pct
        female       0.0         11514 11530.763152     51.831736
        female       1.0         10797 10715.767531     48.168264
         urban       0.0         16392 15854.161689     71.265771
         urban       1.0          5919  6392.368994     28.734229
      division       1.0          1894  1192.500326      5.360388
      division       2.0          4904  5197.752491     23.364328
      division       3.0          4689  5411.883895     24.326867
      division       4.0          2987  2294.547704     10.314182
      division       5.0          1331  1769.402892      7.953613
      division       6.0          2260  2581.909888     11.605899
      division       7.0          2705  2402.019776     10.797278
      division       8.0          1541  1396.513712      6.277445
     water_cat       0.0         11746 11496.470822     51.677590
     water_cat       1.0          5501  4729.404849     21.259067
     water

In [104]:
# Calculate survey-weighted mean, SD, median, and IQR for continuous predictors on df_clean.
cont_vars = ['ub2', 'dds', 'fvs', 'wb4', 'cm11', 'bd9']
univariate_cont = []

for col in cont_vars:
    valid = df_clean[[col, 'chweight']].dropna()
    wq = DescrStatsW(valid[col], weights=valid['chweight'], ddof=0)
    q25, q50, q75 = wq.quantile([0.25, 0.50, 0.75])
    univariate_cont.append({
        'variable': col,
        'unweighted_n': len(valid),
        'weighted_mean': round(wq.mean, 4),
        'weighted_sd': round(wq.std, 4),
        'weighted_median': q50,
        'weighted_iqr': q75 - q25
    })

print(pd.DataFrame(univariate_cont).to_string(index=False))

variable  unweighted_n  weighted_mean  weighted_sd  weighted_median  weighted_iqr
     ub2         22311         1.9844       1.3926              2.0           2.0
     dds          8815         2.3525       1.8323              2.0           3.0
     fvs          8799         2.9130       2.4297              3.0           4.0
     wb4         22052        27.3234       6.2834             26.0           9.0
    cm11         22043         2.1615       1.1086              2.0           2.0
     bd9          6528         4.1181       1.8963              4.0           3.0


In [105]:
# Compute unweighted frequencies and survey-weighted percentages for categorical, binary, and ordinal predictors.
cat_ord_bin_vars = [
    'female', 'urban', 'division', 'water_cat', 'toilet_cat', 'fuel_cat',
    'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'any_illness', 'electricity',
    'edu_level', 'wealth', 'mage_cat', 'parity_cat'
]

univariate_cat = []
total_w = df_model['chweight'].sum()

for col in cat_ord_bin_vars:
    if col in df_model.columns:
        grp = df_model.groupby(col).agg(
            unweighted_n=('chweight', 'count'),
            weighted_n=('chweight', 'sum')
        )
        grp['weighted_pct'] = (grp['weighted_n'] / total_w) * 100
        grp['variable'] = col
        grp['category'] = grp.index
        univariate_cat.append(grp.reset_index(drop=True))

df_uni_cat = pd.concat(univariate_cat, ignore_index=True)[['variable', 'category', 'unweighted_n', 'weighted_n', 'weighted_pct']]
print(df_uni_cat.to_string(index=False))

    variable  category  unweighted_n  weighted_n  weighted_pct
      female       0.0          4514 4584.669893     51.913816
      female       1.0          4256 4246.639799     48.086184
       urban       0.0          6500 6313.837986     71.493790
       urban       1.0          2270 2517.471706     28.506210
    division       1.0           733  457.692748      5.182615
    division       2.0          1916 2081.762363     23.572521
    division       3.0          1859 2224.451159     25.188236
    division       4.0          1189  917.275886     10.386635
    division       5.0           509  646.430127      7.319754
    division       6.0           904 1016.083107     11.505463
    division       7.0          1027  912.246026     10.329680
    division       8.0           633  575.368276      6.515096
   water_cat       0.0          4594 4496.127710     50.911222
   water_cat       1.0          2151 1846.958976     20.913761
   water_cat       2.0          1761 2277.724654     25

### Rao-Scott F-tests

In [106]:
# Compute unweighted counts, Stata-centered 95% CIs, and Rao-Scott F-tests for binary and nominal variables.
bin_nom_vars = [
    'female', 'urban', 'electricity', 'insured', 'mother_married',
    'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'any_illness',
    'division', 'water_cat', 'toilet_cat', 'fuel_cat'
]

bin_nom_summary = []

for col in bin_nom_vars:
    sub = df_clean[['severe_wasting', 'chweight', 'strata', 'hh1', col]].dropna().copy()
    unw_tab = pd.crosstab(sub[col], sub['severe_wasting'])
    est_df, _ = svy_prop_stata_centered(sub, 'severe_wasting', 'chweight', 'strata', 'hh1', col)
    
    comb_tab = pd.DataFrame({
        'Unw_No_Wasting': unw_tab[0],
        'Unw_Severe_Wasting': unw_tab[1],
        'Weighted_Prev_%': est_df['est'].round(2),
        'CI_95_Low_%': est_df['lci'].round(2),
        'CI_95_High_%': est_df['uci'].round(2)
    })
    print(f"\n--- Contingency Table & Design 95% CI: {col} ---")
    print(comb_tab)
    
    chi2_p, f_stat, dof, p_val = rao_scott_chisquare(sub, 'severe_wasting', 'chweight', 'strata', 'hh1', col)
    bin_nom_summary.append({
        'variable': col,
        'pearson_chi2': chi2_p,
        'rao_scott_F': f_stat,
        'df': dof,
        'p_value': p_val
    })

print("\n--- Second-Order Rao-Scott Summary (Binary & Nominal) ---")
print(pd.DataFrame(bin_nom_summary).to_string(index=False))


--- Contingency Table & Design 95% CI: female ---
   Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0           11199                 315             2.89         2.55   
1           10560                 237             2.11         1.82   

   CI_95_High_%  
0          3.27  
1          2.46  

--- Contingency Table & Design 95% CI: urban ---
   Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0           15970                 422             2.64         2.37   
1            5789                 130             2.22         1.82   

   CI_95_High_%  
0          2.93  
1          2.71  

--- Contingency Table & Design 95% CI: electricity ---
   Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0             665                  22             3.33         1.99   
1           21094                 530             2.50         2.27   

   CI_95_High_%  
0          5.54  
1          2.76  

--- Contingency Table & Design 95% CI:

In [107]:
# Generate contingency tables with 95% CIs, Rao-Scott F-tests, and design-adjusted linear trend tests for ordered variables.
ord_vars = ['wealth', 'edu_level', 'ub2', 'parity_cat', 'mage_cat']
ord_trend_summary = []

for col in ord_vars:
    sub = df_clean[['severe_wasting', 'chweight', 'strata', 'hh1', col]].dropna().copy()
    unw_tab = pd.crosstab(sub[col], sub['severe_wasting'])
    est_df, _ = svy_prop_stata_centered(sub, 'severe_wasting', 'chweight', 'strata', 'hh1', col)
    
    comb_tab = pd.DataFrame({
        'Unw_No_Wasting': unw_tab[0],
        'Unw_Severe_Wasting': unw_tab[1],
        'Weighted_Prev_%': est_df['est'].round(2),
        'CI_95_Low_%': est_df['lci'].round(2),
        'CI_95_High_%': est_df['uci'].round(2)
    })
    print(f"\n--- Ordered Contingency Table & 95% CI: {col} ---")
    print(comb_tab)
    
    chi2_p, f_stat, dof, rs_pval = rao_scott_chisquare(sub, 'severe_wasting', 'chweight', 'strata', 'hh1', col)
    m_trend = smf.glm(f"severe_wasting ~ {col}", data=sub, family=sm.families.Binomial(), var_weights=sub['chweight']).fit(cov_type='cluster', cov_kwds={'groups': sub['hh1']})
    or_val = np.exp(m_trend.params[col])
    ci_low, ci_high = np.exp(m_trend.conf_int().loc[col])
    
    ord_trend_summary.append({
        'ordered_variable': col,
        'rao_scott_F': f_stat,
        'rao_scott_p': rs_pval,
        'trend_OR_step': round(or_val, 4),
        'CI_95_Low': round(ci_low, 4),
        'CI_95_High': round(ci_high, 4),
        'trend_z': round(m_trend.tvalues[col], 4),
        'trend_p_value': round(m_trend.pvalues[col], 4)
    })

print("\n--- Design-Adjusted Trend & Rao-Scott Summary (Ordered) ---")
print(pd.DataFrame(ord_trend_summary).to_string(index=False))


--- Ordered Contingency Table & 95% CI: wealth ---
   Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
1            5069                 152             2.98         2.50   
2            4652                 124             2.52         2.04   
3            4349                 106             2.61         2.09   
4            4115                 101             2.42         1.93   
5            3574                  69             2.04         1.57   

   CI_95_High_%  
1          3.56  
2          3.11  
3          3.25  
4          3.02  
5          2.64  

--- Ordered Contingency Table & 95% CI: edu_level ---
     Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0.0            1193                  43             3.57         2.53   
1.0            3867                  99             2.68         2.13   
2.0            5561                 149             2.65         2.21   
3.0            6025                 139             2.18         1

### Univariate logistic regression

In [108]:
# Fit univariable survey-weighted logistic regression and check linearity using quartiles and cubic splines for wb4 and cm11.
cont_vars = ['wb4', 'cm11']
cont_summary = []

for col in cont_vars:
    sub = df_clean[['severe_wasting', 'chweight', 'strata', 'hh1', col]].dropna().copy()
    m_lin = smf.glm(f"severe_wasting ~ {col}", data=sub, family=sm.families.Binomial(), var_weights=sub['chweight']).fit(cov_type='cluster', cov_kwds={'groups': sub['hh1']})
    
    sub['quartile'] = pd.qcut(sub[col], q=4, duplicates='drop')
    unw_q = pd.crosstab(sub['quartile'], sub['severe_wasting'])
    w_q = pd.crosstab(sub['quartile'], sub['severe_wasting'], values=sub['chweight'], aggfunc='sum')
    q_tab = pd.DataFrame({
        'Unw_No_Wasting': unw_q[0],
        'Unw_Severe_Wasting': unw_q[1],
        'Weighted_Prev_%': ((w_q[1] / w_q.sum(axis=1)) * 100).round(2)
    })
    print(f"\n--- Quartile Linearity Check Table: {col} ---")
    print(q_tab)
    
    m_spline = smf.glm(f"severe_wasting ~ bs({col}, df=3)", data=sub, family=sm.families.Binomial(), var_weights=sub['chweight']).fit(cov_type='cluster', cov_kwds={'groups': sub['hh1']})
    R_spline = np.eye(len(m_spline.params))[1:]
    spline_wald = m_spline.wald_test(R_spline, scalar=True)
    
    cont_summary.append({
        'variable': col,
        'linear_OR': round(np.exp(m_lin.params[col]), 4),
        'linear_wald_z': round(m_lin.tvalues[col], 4),
        'linear_p_value': round(m_lin.pvalues[col], 4),
        'spline_chi2_stat': round(float(spline_wald.statistic), 4),
        'spline_p_value': round(float(spline_wald.pvalue), 4)
    })

print("\n--- Continuous & Count Linearity Summary ---")
print(pd.DataFrame(cont_summary).to_string(index=False))


--- Quartile Linearity Check Table: wb4 ---
                Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
quartile                                                           
(14.999, 22.0]            5484                 153             2.69
(22.0, 26.0]              5380                 116             1.98
(26.0, 31.0]              5307                 146             2.80
(31.0, 49.0]              5337                 129             2.58

--- Quartile Linearity Check Table: cm11 ---
               Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
quartile                                                          
(-0.001, 1.0]            6601                 170             2.55
(1.0, 2.0]               8084                 194             2.30
(2.0, 3.0]               4473                 115             2.63
(3.0, 9.0]               2341                  65             2.94

--- Continuous & Count Linearity Summary ---
variable  linear_OR  linear_wald_z  linear_p_value  s

### Rao-Scott $\chi^2$

In [109]:
# Filter subpopulation to children aged 0-1 (young == 1) and run Rao-Scott chi-square and Stata-centered 95% CIs on infant variables.
df_young = df_clean[df_clean['young'] == 1].copy()

young_cat_vars = [
    'fg_grain', 'fg_legume', 'fg_dairy', 'fg_flesh', 'fg_egg',
    'fg_vita', 'fg_ofv', 'bf_status', 'bottle', 'anc_cat'
]
young_cat_summary = []

for col in young_cat_vars:
    sub = df_young[['severe_wasting', 'chweight', 'strata', 'hh1', col]].dropna().copy()
    unw_tab = pd.crosstab(sub[col], sub['severe_wasting'])
    est_df, _ = svy_prop_stata_centered(sub, 'severe_wasting', 'chweight', 'strata', 'hh1', col)
    
    comb_tab = pd.DataFrame({
        'Unw_No_Wasting': unw_tab[0],
        'Unw_Severe_Wasting': unw_tab[1],
        'Weighted_Prev_%': est_df['est'].round(2),
        'CI_95_Low_%': est_df['lci'].round(2),
        'CI_95_High_%': est_df['uci'].round(2)
    })
    print(f"\n--- Ages 0-1 Contingency Table & 95% CI: {col} ---")
    print(comb_tab)
    
    chi2_p, f_stat, dof, p_val = rao_scott_chisquare(sub, 'severe_wasting', 'chweight', 'strata', 'hh1', col)
    young_cat_summary.append({
        'infant_variable': col,
        'pearson_chi2': chi2_p,
        'rao_scott_F': f_stat,
        'df': dof,
        'p_value': p_val
    })

print("\n--- Ages 0-1 Rao-Scott Summary ---")
print(pd.DataFrame(young_cat_summary).to_string(index=False))


--- Ages 0-1 Contingency Table & 95% CI: fg_grain ---
     Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0.0            2887                 136             4.43         3.68   
1.0            5698                 161             2.73         2.29   

     CI_95_High_%  
0.0          5.34  
1.0          3.26  

--- Ages 0-1 Contingency Table & 95% CI: fg_legume ---
     Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0.0            7428                 265             3.47         3.04   
1.0            1160                  32             2.36         1.54   

     CI_95_High_%  
0.0          3.96  
1.0          3.60  

--- Ages 0-1 Contingency Table & 95% CI: fg_dairy ---
     Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%  CI_95_Low_%  \
0.0            6077                 225             3.53         3.05   
1.0            2493                  71             2.76         2.14   

     CI_95_High_%  
0.0          4.09  
1.0          3

### Wald tests for dds, fvs, ordered anc

In [110]:
# Run design-adjusted trend and univariable Wald tests for dietary diversity (dds), food variety (fvs), and ordered ANC visits within young == 1.
young_score_vars = ['dds', 'fvs', 'anc_cat']
young_score_summary = []

for col in young_score_vars:
    sub = df_young[['severe_wasting', 'chweight', 'strata', 'hh1', col]].dropna().copy()
    m_score = smf.glm(f"severe_wasting ~ {col}", data=sub, family=sm.families.Binomial(), var_weights=sub['chweight']).fit(cov_type='cluster', cov_kwds={'groups': sub['hh1']})
    or_val = np.exp(m_score.params[col])
    ci_low, ci_high = np.exp(m_score.conf_int().loc[col])
    
    young_score_summary.append({
        'infant_score_var': col,
        'trend_OR': round(or_val, 4),
        'CI_95_Low': round(ci_low, 4),
        'CI_95_High': round(ci_high, 4),
        'wald_z': round(m_score.tvalues[col], 4),
        'p_value': round(m_score.pvalues[col], 4)
    })

print("\n--- Ages 0-1 Score & Trend Summary (dds, fvs, anc_cat) ---")
print(pd.DataFrame(young_score_summary).to_string(index=False))


--- Ages 0-1 Score & Trend Summary (dds, fvs, anc_cat) ---
infant_score_var  trend_OR  CI_95_Low  CI_95_High  wald_z  p_value
             dds    0.8643     0.7958      0.9387 -3.4608   0.0005
             fvs    0.9083     0.8483      0.9724 -2.7645   0.0057
         anc_cat    0.7097     0.5722      0.8804 -3.1193   0.0018


## Multivariate

### Survey-weighted logistic regression (Age: 0-4)

In [111]:
# Fit multivariable survey-weighted logistic regression for all children aged 0-4 (excluding infant-only variables so N=22,311 is preserved).
features_all_ages = [
    'ub2', 'female', 'diarrhoea_2w', 'cough_2w', 'breathing_2w',
    'edu_level', 'mage_cat', 'parity_cat', 'water_cat',
    'toilet_cat', 'fuel_cat', 'urban', 'wealth', 'division'
]
df_multivar_all = df_clean.dropna(subset=features_all_ages).copy()
df_multivar_all['parity_cat_collapsed'] = df_multivar_all['parity_cat'].replace({0.0: 1.0})

formula_all = "severe_wasting ~ ub2 + female + diarrhoea_2w + cough_2w + breathing_2w + C(edu_level) + C(mage_cat) + C(parity_cat_collapsed) + C(water_cat) + C(toilet_cat) + C(fuel_cat) + urban + C(wealth) + C(division)"

m_multi_all = smf.glm(formula=formula_all, data=df_multivar_all, family=sm.families.Binomial(), var_weights=df_multivar_all['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_multivar_all['hh1']})

res_all = pd.DataFrame({
    'Term': m_multi_all.params.index,
    'AOR': np.exp(m_multi_all.params.values).round(4),
    'CI_95_Low': np.exp(m_multi_all.conf_int()[0].values).round(4),
    'CI_95_High': np.exp(m_multi_all.conf_int()[1].values).round(4),
    'Wald_z': m_multi_all.tvalues.values.round(4),
    'p_value': m_multi_all.pvalues.values.round(4)
})
print(f"--- Multivariable Model: All Children 0-4 (N = {len(df_multivar_all)}) ---")
print(res_all.to_string(index=False))

--- Multivariable Model: All Children 0-4 (N = 22017) ---
                          Term    AOR  CI_95_Low  CI_95_High  Wald_z  p_value
                     Intercept 0.0486     0.0218      0.1085 -7.3760   0.0000
           C(edu_level)[T.1.0] 0.8099     0.5239      1.2521 -0.9484   0.3429
           C(edu_level)[T.2.0] 0.8493     0.5577      1.2933 -0.7614   0.4464
           C(edu_level)[T.3.0] 0.6886     0.4321      1.0973 -1.5694   0.1166
           C(edu_level)[T.4.0] 0.8317     0.4972      1.3914 -0.7018   0.4828
           C(edu_level)[T.5.0] 0.7923     0.4408      1.4242 -0.7782   0.4365
            C(mage_cat)[T.2.0] 1.3148     0.8923      1.9375  1.3838   0.1664
            C(mage_cat)[T.3.0] 1.4002     0.9192      2.1331  1.5674   0.1170
            C(mage_cat)[T.4.0] 1.7521     1.0731      2.8607  2.2420   0.0250
            C(mage_cat)[T.5.0] 1.8170     1.0835      3.0473  2.2638   0.0236
C(parity_cat_collapsed)[T.2.0] 0.8276     0.6285      1.0897 -1.3482   0.1776
C(pari

### Survey-weighted logistic regression (Age: 0-1)

In [112]:
# Fit multivariable survey-weighted logistic regression specifically for infants aged 0-1 (young == 1) including feeding and ANC predictors.
features_young = features_all_ages + ['dds', 'anc_cat', 'bf_status', 'bottle', 'fg_grain', 'fg_flesh']
df_multivar_young = df_young.dropna(subset=features_young).copy()
df_multivar_young['parity_cat_collapsed'] = df_multivar_young['parity_cat'].replace({0.0: 1.0})

formula_young = "severe_wasting ~ ub2 + female + diarrhoea_2w + cough_2w + breathing_2w + dds + fg_grain + fg_flesh + C(anc_cat) + C(bf_status) + bottle + C(edu_level) + C(mage_cat) + C(parity_cat_collapsed) + C(water_cat) + C(toilet_cat) + C(fuel_cat) + urban + C(wealth) + C(division)"

m_multi_young = smf.glm(formula=formula_young, data=df_multivar_young, family=sm.families.Binomial(), var_weights=df_multivar_young['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_multivar_young['hh1']})

res_young = pd.DataFrame({
    'Term': m_multi_young.params.index,
    'AOR': np.exp(m_multi_young.params.values).round(4),
    'CI_95_Low': np.exp(m_multi_young.conf_int()[0].values).round(4),
    'CI_95_High': np.exp(m_multi_young.conf_int()[1].values).round(4),
    'Wald_z': m_multi_young.tvalues.values.round(4),
    'p_value': m_multi_young.pvalues.values.round(4)
})
print(f"\n--- Multivariable Model: Infants Aged 0-1 (N = {len(df_multivar_young)}) ---")
print(res_young.to_string(index=False))


--- Multivariable Model: Infants Aged 0-1 (N = 8713) ---
                          Term    AOR  CI_95_Low  CI_95_High  Wald_z  p_value
                     Intercept 0.0512     0.0112      0.2348 -3.8244   0.0001
             C(anc_cat)[T.1.0] 0.8653     0.5244      1.4279 -0.5661   0.5713
             C(anc_cat)[T.2.0] 0.6555     0.3804      1.1297 -1.5207   0.1283
           C(bf_status)[T.1.0] 1.1178     0.3658      3.4153  0.1954   0.8451
           C(bf_status)[T.2.0] 1.7127     0.6673      4.3963  1.1188   0.2632
           C(edu_level)[T.1.0] 0.9954     0.5478      1.8090 -0.0151   0.9880
           C(edu_level)[T.2.0] 1.1082     0.6144      1.9991  0.3415   0.7328
           C(edu_level)[T.3.0] 0.9110     0.4872      1.7033 -0.2921   0.7702
           C(edu_level)[T.4.0] 1.1012     0.5654      2.1449  0.2835   0.7768
           C(edu_level)[T.5.0] 0.9520     0.4305      2.1050 -0.1215   0.9033
            C(mage_cat)[T.2.0] 1.2913     0.7964      2.0937  1.0366   0.2999
      

### Survey-Weighted Multivariable Logistic Regression & Joint Wald Tests

In [120]:
# Compute robust joint Wald tests for multi-level categorical factors using explicit scalar type conversion.
joint_tests = {}
for term in ['edu_level', 'mage_cat', 'parity_cat', 'water_cat', 'toilet_cat', 'fuel_cat', 'wealth', 'division']:
    params_idx = [i for i, name in enumerate(glm_main.params.index) if f"C({term})" in name]
    if params_idx:
        constraint_matrix = np.eye(len(glm_main.params))[params_idx]
        wald_res = glm_main.wald_test(constraint_matrix, scalar=True)
        stat_val = np.squeeze(wald_res.statistic)
        pval_val = np.squeeze(wald_res.pvalue)
        joint_tests[term] = {
            'F_stat': float(stat_val.item() if hasattr(stat_val, 'item') else stat_val),
            'p_value': float(pval_val.item() if hasattr(pval_val, 'item') else pval_val)
        }
print("\n--- Joint Wald Tests for Multi-Level Factors ---")
print(pd.DataFrame.from_dict(joint_tests, orient='index').to_string())


--- Joint Wald Tests for Multi-Level Factors ---
                F_stat       p_value
edu_level     3.448330  6.312200e-01
mage_cat      6.503050  1.645983e-01
parity_cat  117.885390  1.510947e-24
water_cat     0.453546  9.289720e-01
toilet_cat    5.289827  1.517639e-01
fuel_cat      5.194198  2.679456e-01
wealth        2.567726  6.325509e-01
division     17.004164  1.736935e-02


### ROC-AUC (Model Calibration)

In [114]:
# Calculate survey-weighted ROC-AUC and evaluate calibration performance for the multivariable model.
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve

preds = glm_main.predict(df_multi)
weighted_auc = roc_auc_score(df_multi['severe_wasting'].astype(int), preds, sample_weight=df_multi['chweight'])
print(f"Survey-Weighted ROC-AUC: {weighted_auc:.4f}")

prob_true, prob_pred = calibration_curve(df_multi['severe_wasting'].astype(int), preds, n_bins=10, strategy='quantile')
calib_df = pd.DataFrame({'Mean_Predicted': prob_pred, 'Fraction_True': prob_true})
print("\n--- Calibration Curve Deciles ---")
print(calib_df.to_string(index=False))

Survey-Weighted ROC-AUC: 0.6352

--- Calibration Curve Deciles ---
 Mean_Predicted  Fraction_True
       0.009937       0.012262
       0.013780       0.016803
       0.016404       0.019991
       0.018782       0.014532
       0.021270       0.021344
       0.024017       0.022263
       0.027226       0.025886
       0.031269       0.028169
       0.037254       0.034060
       0.054211       0.051317


### Poisson regression for adjusted Prevalence Ratios (PRs)

In [115]:
# Estimate adjusted prevalence ratios (PRs) using a survey-weighted Poisson GLM with robust cluster standard errors.
poisson_model = smf.glm(formula=main_formula, data=df_multi, family=sm.families.Poisson(link=sm.families.links.Log()), var_weights=df_multi['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_multi['hh1']})

pr_results = pd.DataFrame({
    'Term': poisson_model.params.index,
    'APR': np.exp(poisson_model.params.values).round(4),
    'CI_Low': np.exp(poisson_model.conf_int()[0].values).round(4),
    'CI_High': np.exp(poisson_model.conf_int()[1].values).round(4),
    'p_value': poisson_model.pvalues.values.round(4)
})
print(pr_results.to_string(index=False))

                Term          APR  CI_Low      CI_High  p_value
           Intercept 0.000000e+00     NaN          NaN      NaN
 C(edu_level)[T.1.0] 8.182000e-01  0.5385 1.243300e+00   0.3473
 C(edu_level)[T.2.0] 8.541000e-01  0.5700 1.279800e+00   0.4447
 C(edu_level)[T.3.0] 6.953000e-01  0.4437 1.089700e+00   0.1129
 C(edu_level)[T.4.0] 8.332000e-01  0.5075 1.367900e+00   0.4707
 C(edu_level)[T.5.0] 7.941000e-01  0.4506 1.399600e+00   0.4252
  C(mage_cat)[T.2.0] 1.309600e+00  0.8988 1.908400e+00   0.1602
  C(mage_cat)[T.3.0] 1.399200e+00  0.9292 2.106900e+00   0.1077
  C(mage_cat)[T.4.0] 1.743100e+00  1.0820 2.808100e+00   0.0224
  C(mage_cat)[T.5.0] 1.809300e+00  1.0940 2.992200e+00   0.0209
C(parity_cat)[T.1.0] 3.806177e+08  0.0000 1.304908e+88   0.8325
C(parity_cat)[T.2.0] 3.135364e+08     NaN          NaN      NaN
C(parity_cat)[T.3.0] 3.078782e+08     NaN          NaN      NaN
C(parity_cat)[T.4.0] 2.976984e+08  0.0000 1.736620e+73   0.7976
   C(water_cat)[T.1] 9.113000e-01  0.687

### Interaction Tests

In [121]:
# Test key epidemiological interaction effects safely using robust coefficient z-tests.
df_multi['young_wealth'] = df_multi['young'] * df_multi['wealth']
df_multi['young_urban'] = df_multi['young'] * df_multi['urban']
df_multi['female_young'] = df_multi['female'] * df_multi['young']

inter_formula = main_formula + " + young*wealth + young*urban + female*young"
glm_inter = smf.glm(formula=inter_formula, data=df_multi, family=sm.families.Binomial(), var_weights=df_multi['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_multi['hh1']})

inter_output = []
for name, param in glm_inter.params.items():
    if ':' in name or '*' in name:
        se = glm_inter.bse[name]
        z_val = glm_inter.tvalues[name]
        p_val = glm_inter.pvalues[name]
        inter_output.append({'Interaction_Term': name, 'Coef': round(param, 4), 'SE': round(se, 4), 'z': round(z_val, 4), 'p_value': round(p_val, 4)})

print(pd.DataFrame(inter_output).to_string(index=False))

Interaction_Term          Coef  SE   z  p_value
    young:wealth -4.949391e+13 NaN NaN      NaN
     young:urban  1.455605e+14 NaN NaN      NaN
    female:young  1.878925e+14 NaN NaN      NaN


### Multi-level mied-effects

In [ ]:
# Fit a multilevel mixed-effects logistic regression with random intercepts for cluster, district, and division to quantify spatial variance.
import statsmodels.formula.api as smf_mixed

mixed_formula = "severe_wasting ~ ub2 + female + diarrhoea_2w + cough_2w + breathing_2w + urban"
mixed_model = smf_mixed.mixedglm(formula=mixed_formula, data=df_multi, family=sm.families.Binomial(), groups=df_multi['division']).fit()
print(mixed_model.summary())

### Regularization

In [117]:
# Fit an L2-penalized survey-weighted logistic regression to handle sparse categories and stabilize coefficients.
from sklearn.linear_model import LogisticRegression

X_pen = pd.get_dummies(df_multi[['ub2', 'female', 'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'urban', 'wealth', 'division']], drop_first=True)
y_pen = df_multi['severe_wasting'].astype(int).to_numpy()
w_pen = df_multi['chweight'].to_numpy()

ridge_clf = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', random_state=42)
ridge_clf.fit(X_pen, y_pen, sample_weight=w_pen)
ridge_coefs = pd.DataFrame({'Feature': X_pen.columns, 'Coefficient': ridge_clf.coef_[0]}).sort_values(by='Coefficient', ascending=False)
print(ridge_coefs.to_string(index=False))

     Feature  Coefficient
diarrhoea_2w     0.347092
    cough_2w     0.177105
    division    -0.054097
      wealth    -0.085563
       urban    -0.091314
breathing_2w    -0.104720
         ub2    -0.199277
      female    -0.315320


### Infant sub-population model (Age: 0-1)

In [118]:
# Fit a multivariable survey-weighted logistic regression specifically for infants aged 0-1, incorporating feeding and ANC factors.
df_infants = df_clean[(df_clean['young'] == 1)].copy()
infant_formula = "severe_wasting ~ ub2 + female + diarrhoea_2w + cough_2w + breathing_2w + dds + fg_grain + fg_flesh + C(anc_cat) + C(bf_status) + bottle + C(edu_level) + C(mage_cat) + C(parity_cat) + C(water_cat) + C(toilet_cat) + C(fuel_cat) + urban + C(wealth) + C(division)"

df_infants_multi = df_infants.dropna(subset=['ub2', 'female', 'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'dds', 'anc_cat', 'bf_status', 'bottle', 'edu_level', 'mage_cat', 'parity_cat', 'water_cat', 'toilet_cat', 'fuel_cat', 'urban', 'wealth', 'division']).copy()

glm_infant = smf.glm(formula=infant_formula, data=df_infants_multi, family=sm.families.Binomial(), var_weights=df_infants_multi['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_infants_multi['hh1']})

infant_results = pd.DataFrame({
    'Term': glm_infant.params.index,
    'AOR': np.exp(glm_infant.params.values).round(4),
    'CI_Low': np.exp(glm_infant.conf_int()[0].values).round(4),
    'CI_High': np.exp(glm_infant.conf_int()[1].values).round(4),
    'z': glm_infant.tvalues.values.round(4),
    'p_value': glm_infant.pvalues.values.round(4)
})
print(infant_results.to_string(index=False))

                Term    AOR  CI_Low  CI_High       z  p_value
           Intercept 0.0512  0.0112   0.2348 -3.8244   0.0001
   C(anc_cat)[T.1.0] 0.8653  0.5244   1.4279 -0.5661   0.5713
   C(anc_cat)[T.2.0] 0.6555  0.3804   1.1297 -1.5207   0.1283
 C(bf_status)[T.1.0] 1.1178  0.3658   3.4153  0.1954   0.8451
 C(bf_status)[T.2.0] 1.7127  0.6673   4.3963  1.1188   0.2632
 C(edu_level)[T.1.0] 0.9954  0.5478   1.8090 -0.0151   0.9880
 C(edu_level)[T.2.0] 1.1082  0.6144   1.9991  0.3415   0.7328
 C(edu_level)[T.3.0] 0.9110  0.4872   1.7033 -0.2921   0.7702
 C(edu_level)[T.4.0] 1.1012  0.5654   2.1449  0.2835   0.7768
 C(edu_level)[T.5.0] 0.9520  0.4305   2.1050 -0.1215   0.9033
  C(mage_cat)[T.2.0] 1.2913  0.7964   2.0937  1.0366   0.2999
  C(mage_cat)[T.3.0] 1.4425  0.8269   2.5161  1.2906   0.1969
  C(mage_cat)[T.4.0] 2.0904  1.0764   4.0596  2.1774   0.0295
  C(mage_cat)[T.5.0] 2.3619  1.1337   4.9207  2.2950   0.0217
C(parity_cat)[T.2.0] 0.7497  0.5063   1.1101 -1.4385   0.1503
C(parity

### Non-infant sub-population model (Age: 2-4)

In [122]:
# Fit a multivariable survey-weighted logistic regression specifically for non-infant children aged 2-4 (young == 0) using age-appropriate predictors.
df_older = df_clean[df_clean['young'] == 0].copy()
older_formula = "severe_wasting ~ ub2 + female + diarrhoea_2w + cough_2w + breathing_2w + C(edu_level) + C(mage_cat) + C(parity_cat) + C(water_cat) + C(toilet_cat) + C(fuel_cat) + urban + C(wealth) + C(division)"

df_older_multi = df_older.dropna(subset=['ub2', 'female', 'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'edu_level', 'mage_cat', 'parity_cat', 'water_cat', 'toilet_cat', 'fuel_cat', 'urban', 'wealth', 'division']).copy()

glm_older = smf.glm(formula=older_formula, data=df_older_multi, family=sm.families.Binomial(), var_weights=df_older_multi['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_older_multi['hh1']})

older_results = pd.DataFrame({
    'Term': glm_older.params.index,
    'AOR': np.exp(glm_older.params.values).round(4),
    'CI_Low': np.exp(glm_older.conf_int()[0].values).round(4),
    'CI_High': np.exp(glm_older.conf_int()[1].values).round(4),
    'z': glm_older.tvalues.values.round(4),
    'p_value': glm_older.pvalues.values.round(4)
})
print(f"\n--- Multivariable Model: Non-Infants Aged 2-4 (N = {len(df_older_multi)}) ---")
print(older_results.to_string(index=False))


--- Multivariable Model: Non-Infants Aged 2-4 (N = 13174) ---
                Term          AOR  CI_Low       CI_High       z  p_value
           Intercept 0.000000e+00  0.0000           inf -0.0539   0.9570
 C(edu_level)[T.1.0] 7.088000e-01  0.3714  1.352600e+00 -1.0440   0.2965
 C(edu_level)[T.2.0] 6.913000e-01  0.3637  1.313900e+00 -1.1267   0.2599
 C(edu_level)[T.3.0] 5.522000e-01  0.2677  1.138900e+00 -1.6078   0.1079
 C(edu_level)[T.4.0] 6.957000e-01  0.3073  1.574900e+00 -0.8704   0.3841
 C(edu_level)[T.5.0] 7.617000e-01  0.3215  1.804800e+00 -0.6184   0.5363
  C(mage_cat)[T.2.0] 1.746900e+00  0.7714  3.956100e+00  1.3376   0.1810
  C(mage_cat)[T.3.0] 1.739800e+00  0.7569  3.998800e+00  1.3041   0.1922
  C(mage_cat)[T.4.0] 2.009900e+00  0.8279  4.879400e+00  1.5425   0.1229
  C(mage_cat)[T.5.0] 2.055700e+00  0.8181  5.165400e+00  1.5328   0.1253
C(parity_cat)[T.1.0] 3.691662e+08  0.0000           inf  0.0404   0.9678
C(parity_cat)[T.2.0] 3.205767e+08  0.0000           inf  0.05